In [40]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import csv

## check from raw query to abstracts

In [32]:
import glob
import pandas as pd
import os

def extract_pmids_from_file(file_path, separator="|||"):
    """
    Reads a file and extracts the first column (PMID) as a set.
    """
    pmid_set = set()
    pmid_set_with_abstract = set()
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            fields = line.strip().split(separator)
            if fields:  # Ensure line is not empty
                try:
                    abstract = fields[4]
                    pmid = fields[0]
                    pmid_set.add(pmid)  # Assuming PMID is the first column
                    if abstract != "N/A":
                        pmid_set_with_abstract.add(pmid)
                        
                except:
                    print(f"SKIPPING line: {pmid} FROM {file_path}, {fields}")
                    continue
    return pmid_set, pmid_set_with_abstract

def save_pmids_with_data(files_path, output_pmids_with_content, output_pmids_with_abstract, separator='|||', target_pmids=None):
    """
    Reads and filters multiple large files in a folder efficiently.
    """
    for path in [output_pmids_with_content, output_pmids_with_abstract]:
        if os.path.exists(path):
            os.remove(path)
            print(f"Deleted existing file: {path}")
    
    data_folders = [
        'round_1', 'round_2', 'round_3', 'round_4',
        'round_5', 'round_6', 'round_7', 'round_8'
    ]
    file_list = []
    for folder in data_folders:
        print(f"Processing {folder}")
        folder_path = os.path.join(files_path, folder)
        file_list.extend(glob.glob(os.path.join(folder_path, '*.txt')))
    
    for file_path in file_list:
        if os.path.getsize(file_path) == 0:
            continue  # Skip empty files
        fetched_pmids, pmid_set_with_abstract = extract_pmids_from_file(file_path)
    
        # Save 
        with open(output_pmids_with_content, 'a', encoding='utf-8') as out_file:
            for pmid in sorted(fetched_pmids):  # Sort to maintain order
                out_file.write(f"{pmid}\n")
                
        with open(output_pmids_with_abstract, 'a', encoding='utf-8') as out_file:
            for pmid in sorted(pmid_set_with_abstract):  # Sort to maintain order
                out_file.write(f"{pmid}\n")
       
       
def check_pmids_missing_data(pmids_with_content_file, pmids_with_abstract, all_expected_pmids_file):
    """
    Compare PMID coverage between:
      - all expected PMIDs
      - those with full content
      - those with abstracts

    Prints total counts, missing counts, and percentage coverage for each.
    """
    # --- Load sets ---
    def load_pmids(path):
        with open(path) as f:
            return set(map(int, filter(str.isdigit, f.read().splitlines())))

    with_data_pmids = load_pmids(pmids_with_content_file)
    with_abstract_pmids = load_pmids(pmids_with_abstract)
    all_target_pmids = load_pmids(all_expected_pmids_file)

    # --- Compute missing ---
    missing_data_pmids = all_target_pmids - with_data_pmids
    missing_abstract_pmids = all_target_pmids - with_abstract_pmids

    # --- Counts ---
    total = len(all_target_pmids)
    have_data = len(with_data_pmids)
    have_abstract = len(with_abstract_pmids)
    missing_data = len(missing_data_pmids)
    missing_abs = len(missing_abstract_pmids)

    # --- Percentages ---
    pct_data = have_data / total * 100 if total else 0
    pct_abs = have_abstract / total * 100 if total else 0

    # --- Print summary ---
    print(f"PMID Coverage Summary")
    print(f"- Expected total: {total}")
    print(f"- With full content: {have_data} ({pct_data:.1f}%)")
    print(f"- With abstract:     {have_abstract} ({pct_abs:.1f}%)")
    print(f"- Missing content:   {missing_data}")
    print(f"- Missing abstracts: {missing_abs}")

    output_file_path = f'/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/01_pubmed_query_neuro/data/pubmed_queries/missing_pmids_{len(missing_data_pmids)}.txt'
    with open(output_file_path, 'w') as file:
        file.writelines(f"{pmid}\n" for pmid in missing_data_pmids)
        
    output_file_path_abstr = f'/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/01_pubmed_query_neuro/data/pubmed_queries/missing_pmids_abstract_{len(missing_abstract_pmids)}.txt'
    with open(output_file_path_abstr, 'w') as file:
        file.writelines(f"{pmid}\n" for pmid in missing_abstract_pmids)
        


In [33]:
pubmed_content_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/01_pubmed_query_neuro/data/full_pubmed_raw/"
output_pmids_with_content = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/01_pubmed_query_neuro/data/full_pubmed_raw/pmids_with_data.txt"
output_pmids_with_abstract = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/01_pubmed_query_neuro/data/full_pubmed_raw/pmids_with_abstract.txt"
#save_pmids_with_data(pubmed_content_path, output_pmids_with_content, output_pmids_with_abstract)

all_expected_pmids_file = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline//01_pubmed_query_neuro/data/pubmed_queries/union_all_queries_pmids_21704840.txt"
check_pmids_missing_data(output_pmids_with_content, output_pmids_with_abstract, all_expected_pmids_file)

PMID Coverage Summary
- Expected total: 21704840
- With full content: 21684927 (99.9%)
- With abstract:     18343356 (84.5%)
- Missing content:   19913
- Missing abstracts: 3361484


## check studytypeteller animal studies

In [35]:
original_study_type_teller = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/02_animal_study_classification/model_predictions/all_animal_studies_clean_complete.csv"
original_study_type_teller_df = pd.read_csv(original_study_type_teller)
original_study_type_teller_df.head()


,PMID,label_id,label_name,confidence
0,14051997,1,Animal,0.831374
1,14052080,1,Animal,0.831373
2,14052235,1,Animal,0.831374
3,14052236,1,Animal,0.831373
4,14052238,1,Animal,0.831373


In [36]:
original_study_type_teller_df.shape

(6002827, 4)

In [55]:
original_study_type_teller_df.shape[0]/21684927

0.2768202539948601

In [41]:
folder = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/02_animal_study_classification/data/animal_studies_for_ner"

unique_pmids = set()
total_rows = 0

for file_path in glob.glob(os.path.join(folder, "*.csv")):
    with open(file_path, newline='', encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pmid = row.get("PMID")
            if pmid and pmid.isdigit():
                unique_pmids.add(int(pmid))
            total_rows += 1

print("CSV Summary")
print(f"- Files processed: {len(glob.glob(os.path.join(folder, '*.csv')))}")
print(f"- Total rows (including duplicates): {total_rows:,}")
print(f"- Unique PMIDs across all files: {len(unique_pmids):,}")

CSV Summary
- Files processed: 617
- Total rows (including duplicates): 6,160,794
- Unique PMIDs across all files: 6,002,827


In [42]:
file_ner_non_empty = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/03_IE_ner/data/animal_studies_with_drug_disease/filtered_df_non_empty_595768_PMIDs.csv"
file_ner_non_empty_df = pd.read_csv(file_ner_non_empty)
file_ner_non_empty_df

,PMID
0,31733831
1,31733833
2,31733925
3,31733940
4,31734027
...,...
595763,57929
595764,57940
595765,58073
595766,58789


In [51]:
excl_pmids_study_types = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/03_IE_ner/check_study_type/animal_studies_publication_type_counts.csv")
excl_pmids_study_types

,type,count
0,review,27801
1,clinical trial,6207
2,case report,3897
3,randomized controlled trial,3556


In [53]:
excl_pmids_study_types['count'].sum()

41461

In [54]:
len(file_ner_non_empty_df) - excl_pmids_study_types['count'].sum()

554307

In [9]:
target_file = "../03_IE_ner/data/animal_studies_with_drug_disease/animal_studies_metadata_after_stype_filter_554307_PMIDs.csv" #"03_IE_ner/data/animal_studies_with_drug_disease/disease_filtered/all_filtered_disease_pmids_47059.csv"
target_df = pd.read_csv(target_file)

In [10]:
target_df.head()

,PMID
0,31733831
1,31733833
2,31733925
3,31733940
4,31734027


In [6]:
# Input and output file paths
input_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/07_full_text_retrieval/materials_methods/combined/combined_materials_methods.jsonl"

extracted = []

# Read JSONL and extract required fields
with open(input_path, "r", encoding="utf-8") as infile:
    for line in infile:
        record = json.loads(line)
        reduced = {
            "PMID": record.get("PMID"),
            "source_label": record.get("source_label")
        }
        pmids_from_jsonl.append(str(record.get("PMID")))
        extracted.append(reduced)

# Optional: display as table
df = pd.DataFrame(extracted)
print(df.head())


       PMID source_label
0  10063484       cadmus
1  10068158       cadmus
2  10068163       cadmus
3  10068171       cadmus
4  10068174       cadmus


In [30]:
df.to_csv("cadmus/successfully_retrieved_PMIDs_UoZ.csv", index=False)

In [8]:
counts = df.groupby("source_label")["PMID"].nunique().reset_index()
counts = counts.rename(columns={"PMID": "PMID_count"})

# Print table
print(counts)

  source_label  PMID_count
0    bioc_json      139350
1       cadmus      201068


In [12]:
pmids_from_jsonl = set(df['PMID'])
len(pmids_from_jsonl)

340418

In [23]:
target_df["PMID"] = target_df["PMID"].astype(str).str.strip()

# --- Find overlaps and missing ---
target_df["in_jsonl"] = target_df["PMID"].isin(pmids_from_jsonl)

total_target = len(target_df)
found_count = target_df["in_jsonl"].sum()
missing_count = (~target_df["in_jsonl"]).sum()

print(f"Target {total_target}")
print(f"Found {found_count} target PMIDs in JSONL")
print(f"Missing {missing_count} target PMIDs")
print(f"Percentage found {round(found_count/total_target, 2)}")

# Save missing ones
missing_df = target_df[~target_df["in_jsonl"]]

Target 554307
Found 340895 target PMIDs in JSONL
Missing 213412 target PMIDs
Percentage found 0.61


In [24]:
missing_df

,PMID,in_jsonl
19,31734396,False
23,31734690,False
26,31734822,False
32,31735084,False
33,31735085,False
...,...,...
554299,56906,False
554301,57617,False
554303,57929,False
554305,58789,False


In [27]:
missing_pmids = missing_df[["PMID"]].copy()

# Shuffle the rows
missing_pmids = missing_pmids.sample(frac=1, random_state=42).reset_index(drop=True)

# Batch size
batch_size = 50_000

# Save in chunks
for i in range(0, len(missing_pmids), batch_size):
    batch = missing_pmids.iloc[i:i + batch_size]
    batch_number = i // batch_size + 1
    output_file = f"cadmus/missing_pmids_batch_{batch_number}.txt"
    batch["PMID"].to_csv(output_file, index=False, header=False)
    print(f"Saved {len(batch)} PMIDs to {output_file}")

Saved 50000 PMIDs to cadmus/missing_pmids_batch_1.txt
Saved 50000 PMIDs to cadmus/missing_pmids_batch_2.txt
Saved 50000 PMIDs to cadmus/missing_pmids_batch_3.txt
Saved 50000 PMIDs to cadmus/missing_pmids_batch_4.txt
Saved 13412 PMIDs to cadmus/missing_pmids_batch_5.txt
